In [ ]:
import pathlib
import sys

import networkx as nx
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches
from vizopt import bubblejax

In [ ]:
# Example: Single inclusion tree with leaf nodes (cities) and non-leaf nodes (countries)
# Leaf nodes have fixed "size" attributes, non-leaf nodes have optimizable radii

inclusion_tree = nx.DiGraph()

# Add leaf nodes (cities) with fixed sizes
cities = {
    "Munich": 10,
    "Berlin": 25,
    "Stuttgart": 5,
    "Frankfurt": 5,
    "Nuremberg": 3,
    "Vienna": 15,
    "Salzburg": 5,
}

for city, size in cities.items():
    inclusion_tree.add_node(city, size=size)

# Add non-leaf nodes (countries) - these will have optimizable radii
inclusion_tree.add_node("Germany")
inclusion_tree.add_node("Austria")

# Define containment relationships: (parent, child) means child is contained in parent
inclusion_tree.add_edges_from([
    ("Germany", "Munich"),
    ("Germany", "Berlin"),
    ("Germany", "Stuttgart"),
    ("Germany", "Frankfurt"),
    ("Germany", "Nuremberg"),
    ("Austria", "Vienna"),
    ("Austria", "Salzburg"),
])

# Visualize the tree structure
nx.draw_networkx(inclusion_tree, with_labels=True, node_color='lightblue', 
                 arrows=True, pos=nx.spring_layout(inclusion_tree, seed=42))

In [ ]:
# Optimize the layout using the new function
pos, non_leaf_radii = bubblejax.optimize_circular_layout_with_enclosed_nodes(
    inclusion_tree=inclusion_tree,
    optim_kwargs={"n_iters": 10000, "learning_rate": 1*1e-1}
)

# Visualize the optimized layout
_, ax = plt.subplots(figsize=(10, 10))

# Draw all nodes as circles
for node, node_xy in pos.items():
    # Determine radius: leaf nodes use their "size" attribute, non-leaf use optimized radii
    if inclusion_tree.out_degree(node) == 0:  # leaf node
        radius = inclusion_tree.nodes[node]["size"]
        color = 'lightcoral'  # cities in red
        alpha = 0.6
    else:  # non-leaf node
        radius = non_leaf_radii[node]
        color = 'lightblue'  # countries in blue
        alpha = 0.2
    
    circle = mpatches.Circle(node_xy, radius=radius, color=color, alpha=alpha, ec='black', linewidth=1)
    ax.add_patch(circle)
    ax.text(node_xy[0], node_xy[1], node, color="k", ha="center", va="center", 
            fontsize=10, weight='bold' if inclusion_tree.out_degree(node) > 0 else 'normal')

# Set axis limits
max_radius = max(non_leaf_radii.values())
xmin = min(pos[node][0] - max_radius for node in pos)
xmax = max(pos[node][0] + max_radius for node in pos)
ymin = min(pos[node][1] - max_radius for node in pos)
ymax = max(pos[node][1] + max_radius for node in pos)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal")
ax.set_title("Optimized Bubble Layout: Cities Enclosed in Countries")
ax.grid(True, alpha=0.3)

print(f"Optimized radii for non-leaf nodes: {non_leaf_radii}")

In [ ]:
main_graph = nx.Graph()
main_graph.add_node("Munich", size=10)
main_graph.add_node("Frankfurt", size=5)
main_graph.add_node("Nuremberg", size=3)
main_graph.add_node("Berlin", size=25)
main_graph.add_node("Stuttgart", size=5)
main_graph.add_node("Salzburg", size=5)
main_graph.add_node("Vienna", size=15)
main_graph.add_edges_from([("Munich", "Frankfurt"), ("Munich", "Nuremberg"), ("Frankfurt", "Berlin")])
main_graph.add_edges_from([("Munich", "Stuttgart"), ("Stuttgart", "Berlin")])
main_graph.add_edges_from([("Munich", "Salzburg"), ("Salzburg", "Vienna")])
main_graph.add_edges_from([("Bratislava", "Vienna")])

n_add_nodes = 0
for i_node in range(1, n_add_nodes+1):
    main_graph.add_node(f"Node{i_node}", size=5)
    main_graph.add_edges_from([("Node"+str(i_node), "Munich")])


nx.draw_networkx(main_graph)

_, ax = plt.subplots()
nx.draw_networkx(inclusion_tree)

In [ ]:
pos, enclosing_node_radius_dict = bubblejax.optimize_circular_layout_with_enclosed_and_linked_nodes(graph=main_graph, inclusion_tree=inclusion_tree,
                                                         optim_kwargs={"n_iters": 10000, "learning_rate": 5*1e-1})
pos

In [ ]:
_, ax = plt.subplots(figsize=(9, 8))

for node, node_xy in pos.items():
    if node in main_graph.nodes:
        if "size" in main_graph.nodes[node]:
            radius = main_graph.nodes[node]["size"]
        else:
            radius = 1.0
    else:
        radius = enclosing_node_radius_dict[node]
    circle = mpatches.Circle(node_xy, radius=radius, alpha=0.2)
    ax.add_patch(circle)
    ax.text(node_xy[0], node_xy[1] + 0.9*radius, node, color="k", ha="center", va="center")
for edge in main_graph.edges:
    ax.plot(
        [pos[edge[0]][0], pos[edge[1]][0]],
        [pos[edge[0]][1], pos[edge[1]][1]],
        "k-",
    )
max_node_size = max(main_graph.nodes[node]["size"] for node in main_graph.nodes if "size" in main_graph.nodes[node])
max_node_size = max(enclosing_node_radius_dict.values())
xmin = min(pos[node][0] - max_node_size for node in pos)
xmax = max(pos[node][0] + max_node_size for node in pos)
ymin = min(pos[node][1] - max_node_size for node in pos)
ymax = max(pos[node][1] + max_node_size for node in pos)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
# equal axes
ax.set_aspect("equal")

In [ ]:
inclusion_tree.add_edges_from([("Europe", "Germany"), ("Europe", "Austria")])
inclusion_tree.edges

## Scaling experiment

In [ ]:
n_bubbles = 100


In [ ]:
def make_random_forest(n_bubbles, n_trees=None, seed=42):
    """Create a random inclusion tree (forest) with n_bubbles leaf nodes.

    Args:
        n_bubbles: Total number of leaf (bubble) nodes.
        n_trees: Number of enclosing groups. Defaults to max(1, n_bubbles // 8).
        seed: Random seed for reproducibility.

    Returns:
        NetworkX DiGraph where each edge (root, leaf) means leaf is inside root.
    """
    rng = np.random.default_rng(seed)
    n_trees = n_trees or max(1, n_bubbles // 8)

    tree = nx.DiGraph()
    roots = [f"Group{i}" for i in range(n_trees)]
    for root in roots:
        tree.add_node(root)

    # Assign each bubble to a random group
    assignments = rng.integers(0, n_trees, size=n_bubbles)
    for i in range(n_bubbles):
        size = float(rng.integers(3, 20))
        name = f"Bubble{i}"
        tree.add_node(name, size=size)
        tree.add_edge(roots[assignments[i]], name)

    return tree


In [ ]:
forest = make_random_forest(n_bubbles)

leaf_nodes = [n for n in forest.nodes if forest.out_degree(n) == 0]
group_nodes = [n for n in forest.nodes if forest.out_degree(n) > 0]
print(f"{n_bubbles} bubbles across {len(group_nodes)} groups")

pos, group_radii = bubblejax.optimize_circular_layout_with_enclosed_nodes(
    inclusion_tree=forest,
    optim_kwargs={"n_iters": 2000, "learning_rate": 0.1},
)

# Visualize
fig, ax = plt.subplots(figsize=(12, 12))
colors = plt.cm.tab10.colors

for i, root in enumerate(group_nodes):
    color = colors[i % len(colors)]
    xy = pos[root]
    r = group_radii[root]
    ax.add_patch(mpatches.Circle(xy, radius=r, color=color, alpha=0.12, ec=color, linewidth=2))

for node in leaf_nodes:
    parent = next(iter(forest.predecessors(node)))
    color = colors[group_nodes.index(parent) % len(colors)]
    xy = pos[node]
    r = forest.nodes[node]["size"]
    ax.add_patch(mpatches.Circle(xy, radius=r, color=color, alpha=0.5, ec=color, linewidth=0.5))

all_pos = list(pos.values())
max_r = max(group_radii.values())
margin = max_r * 0.2
ax.set_xlim(min(p[0] for p in all_pos) - margin, max(p[0] for p in all_pos) + margin)
ax.set_ylim(min(p[1] for p in all_pos) - margin, max(p[1] for p in all_pos) + margin)
ax.set_aspect("equal")
ax.set_title(f"Random forest: {n_bubbles} bubbles in {len(group_nodes)} groups")
ax.grid(True, alpha=0.3)
plt.tight_layout()


In [ ]:
import time

n_iters = 500
sizes = [10, 50, 100, 200, 500, 1000]

# Each new n_bubbles changes array shapes, triggering JAX recompilation.
# We time two runs per size: first run (includes JIT compile) and second run (pure compute).
results = []
for n in sizes:
    times_n = []
    for run in range(2):
        forest_n = make_random_forest(n, seed=run)
        t0 = time.perf_counter()
        bubblejax.optimize_circular_layout_with_enclosed_nodes(
            inclusion_tree=forest_n,
            optim_kwargs={"n_iters": n_iters, "learning_rate": 0.1},
        )
        elapsed = time.perf_counter() - t0
        times_n.append(elapsed)
    results.append({"n": n, "compile+run": times_n[0], "run_only": times_n[1]})
    print(f"n={n:4d}  compile+run: {times_n[0]:.2f}s   run_only: {times_n[1]:.2f}s")


In [ ]:
ns = [r["n"] for r in results]
t_compile = [r["compile+run"] for r in results]
t_run = [r["run_only"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, yscale in zip(axes, ["linear", "log"]):
    ax.plot(ns, t_compile, "o--", label="compile + run", color="steelblue")
    ax.plot(ns, t_run, "s-", label="run only (2nd call)", color="tomato")
    ax.set_xlabel("Number of bubbles")
    ax.set_ylabel("Time (s)")
    ax.set_xscale("log")
    ax.set_yscale(yscale)
    ax.set_title(f"Scaling ({n_iters} iters) â€” {yscale} y-axis")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()

# Fit power law on run_only times
log_n = np.log(ns)
log_t = np.log(t_run)
slope, intercept = np.polyfit(log_n, log_t, 1)
n_fit = np.linspace(min(ns), max(ns), 100)
t_fit = np.exp(intercept) * n_fit**slope
axes[1].plot(n_fit, t_fit, "k--", alpha=0.5, label=f"fit: O(n^{slope:.2f})")
axes[1].legend()

plt.suptitle(f"Bubble layout scaling random forest, {n_iters} iters per run")
plt.tight_layout()
print(f"Estimated compute complexity: O(n^{slope:.2f})")
